**ELEGIR DATOS QUE NOS INTERESEN**


HACER HIPÓTESIS. 


CONSULTAR APIS gratuitas y DATA DISPONIBLE A PARTIR DE 1M de filas en adelante para esta clase de proyectos


## **🎯 PENDIENTES**

- SEPARAR EN LOS NOTEBOOKS DEL PROYECTO LA CREACIÓN DE TABLAS (DDL) VS DE LOS ETL (CARGA de los archivos)

![image_1795094761734.png](./image_1795094761734.png "image_1795094761734.png")


### Cómo crear una tabla dimensional 
![image_1795364562603.png](./image_1795364562603.png "image_1795364562603.png")

Entender en profundidad que quiere el negocio respecto a preservar el histórico: 
![image_1795369579614.png](./image_1795369579614.png "image_1795369579614.png")

- **SCD Type 1:** Es como un overwrite. Sobreescribe los datos, NO GUARDA LO ANTERIOR. 

- **SCD Type 2:** Nueva versión del registro: agrega cosas nuevas, pero mantiene lo anterior. Acá es importante.

- **SCD Type 3:** Version inmediatamente anterior y actual. No más. No guarda el hisotrial completo. Comparaciones antes/despues.



In [0]:
%sql 
-- =============================================================
-- SCD Type 1 (Overwrite): actualizar ciudad en dim_zona
-- Si la fila ya existe (match por clave natural), se pisa el valor.
-- No se guarda historial. No hay columnas de fecha ni flag.
-- =============================================================

-- 1) Datos de origen (staging) de ejemplo
CREATE OR REPLACE TEMP VIEW staging_zona_scd1 AS
SELECT * FROM VALUES
    ('capital federal', 'capital federal', 'Buenos Aires'),   -- ya existe: se actualiza ciudad
    ('palermo',         'capital federal', 'CABA'),           -- ya existe: se actualiza ciudad
    ('nueva-zona-test', 'gba norte',       'GBA')             -- no existe: se inserta
AS t(partido, region, ciudad);

-- 2) MERGE SCD1
select * from bootcamp.gold.dim_zona;


MERGE INTO bootcamp.gold.dim_zona AS target --ya lo tenemos de antes
USING staging_zona_scd1 AS source
ON  target.partido = source.partido
AND target.region  = source.region
WHEN MATCHED THEN
    UPDATE SET ciudad = source.ciudad
WHEN NOT MATCHED THEN
    INSERT (partido, region, ciudad)
    VALUES (source.partido, source.region, source.ciudad);

-- 3) Verificación
SELECT * FROM bootcamp.gold.dim_zona
ORDER BY partido, region;